# Phase 1 — Preprocessing & Data Cleaning
**Goal:** Clean the raw dataset, remove noise, balance classes, scale features, and persist all artefacts needed by downstream notebooks.

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
import joblib, os, re

DATA_PATH = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\data\MachineLearningCVE"
SAVE_PATH = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\data"
os.makedirs(SAVE_PATH, exist_ok=True)

## 2. Load Data

In [ ]:
print("Loading data...")
all_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
dataframes = []
for file in all_files:
    df_temp = pd.read_csv(os.path.join(DATA_PATH, file), encoding='utf-8', low_memory=False)
    dataframes.append(df_temp)

df = pd.concat(dataframes, ignore_index=True)
df.columns = df.columns.str.strip()
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 3. Cleaning Steps

### 3.1 Replace Infinite Values

In [ ]:
inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f"Replaced {inf_count} infinite values with NaN")

### 3.2 Drop Rows with Missing Values

In [ ]:
rows_before = len(df)
df.dropna(inplace=True)
print(f"Dropped {rows_before - len(df):,} rows  →  {len(df):,} remaining")

### 3.3 Drop Rows with Negative Flow Duration

In [ ]:
rows_before = len(df)
df = df[df['Flow Duration'] >= 0]
print(f"Dropped {rows_before - len(df):,} rows  →  {len(df):,} remaining")

### 3.4 Drop Duplicate Rows

In [ ]:
rows_before = len(df)
df.drop_duplicates(inplace=True)
print(f"Dropped {rows_before - len(df):,} rows  →  {len(df):,} remaining")

### 3.5 Normalise Label Names

In [ ]:
def clean_label(label):
    label = re.sub(r'Web Attack\s*.+?\s*(Brute Force|XSS|Sql Injection)',
                   r'Web Attack - \1', label)
    return label.strip()

df['Label'] = df['Label'].apply(clean_label)
print(df['Label'].value_counts())

### 3.6 Remove Low-Variance Features

In [ ]:
X = df.drop('Label', axis=1)
y = df['Label']

selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
kept_cols   = X.columns[selector.get_support()].tolist()
dropped_cols = X.columns[~selector.get_support()].tolist()
X = X[kept_cols]

print(f"Features before : {len(kept_cols) + len(dropped_cols)}")
print(f"Features after  : {len(kept_cols)}")
print(f"Dropped         : {dropped_cols}")

## 4. Label Encoding

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label mapping:")
for i, label in enumerate(le.classes_):
    print(f"  {i:2d} → {label}")

## 5. Feature Scaling

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=kept_cols)

print(f"Scaled {X_scaled.shape[1]} features")
print(f"Mean (≈0): {X_scaled.mean().mean():.6f}")
print(f"Std  (≈1): {X_scaled.std().mean():.6f}")

## 6. Save Artefacts

In [ ]:
X_scaled['Label'] = y_encoded
X_scaled.to_parquet(os.path.join(SAVE_PATH, 'clean_data.parquet'), index=False)

joblib.dump(le,        os.path.join(SAVE_PATH, 'label_encoder.pkl'))
joblib.dump(scaler,    os.path.join(SAVE_PATH, 'scaler.pkl'))
joblib.dump(kept_cols, os.path.join(SAVE_PATH, 'feature_columns.pkl'))

print(f"✅ clean_data.parquet  — {X_scaled.shape[0]:,} rows × {X_scaled.shape[1]} columns")
print("✅ label_encoder.pkl")
print("✅ scaler.pkl")
print("✅ feature_columns.pkl")